1. Install Required Libraries
pip install -r requirements.txt

In [1]:
import requests
import xmltodict
import pandas as pd

%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "../..")


In [2]:
from src.config import ROAD_CLOSURES_URL, ROAD_CLOSURE_HEADERS

closureType = "planned"  # Options: "planned", "unplanned"
params = {
    "closureType": closureType,
    "startDateTime": "2026-04-30T00:00:00",
    "endDateTime":   "2026-04-30T23:59:59",
    "modifiedSinceDateTime": "2026-04-30T00:00:00"
}


response = requests.get(ROAD_CLOSURES_URL, headers=ROAD_CLOSURE_HEADERS, params=params)

print("Status Code:", response.status_code)

Status Code: 200


In [3]:
road_closures_dict = xmltodict.parse(response.text)
print(len(road_closures_dict))

1


In [4]:
from src.config import AZURE_CONNECTION_STRING, CONTAINER_ROAD_CLOSURES
from azure.storage.blob import BlobServiceClient

blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_ROAD_CLOSURES)

In [5]:
BATCH_SIZE = 100
UPLOAD_SIZE = 2000

from datetime import datetime

def generate_blob_name(closureType, start_utc):
    if isinstance(start_utc, str):
        start_utc = datetime.fromisoformat(start_utc.replace("Z", "+00:00"))

    timestamp = start_utc.strftime("%Y%m%d_%H%M%S")
    return f"{closureType}_{timestamp}.csv"



In [6]:
from io import StringIO

def upload_to_azure(records, blob_name):
    df = pd.DataFrame(records)

    buffer = StringIO()
    df.to_csv(buffer, index=False)

    blob_client = container_client.get_blob_client(blob_name)
    blob_client.upload_blob(buffer.getvalue(), overwrite=True)

    print(f"Uploaded {len(records)} records → {blob_name}")

In [7]:
import pandas as pd

def ensure_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, dict):
        return [x]
    return []


# --------------------------------------------------------------
# 1. Extract Road Name
# --------------------------------------------------------------
def get_road_name_from_location(loc_ref):

    # Case A: locSingleRoadLinearLocation (most precise)
    single = loc_ref.get("locSingleRoadLinearLocation")
    if single:
        for lw in ensure_list(single.get("linearWithinLinearElement")):
            try:
                name = (
                    lw.get("linearElement", {})
                      .get("locLinearElementByCode", {})
                      .get("roadName")
                )
                if name:
                    return name
            except:
                pass

    # Case B: grouped locations
    group_list = (
        loc_ref.get("locLocationGroupByList", {})
              .get("locationContainedInGroup", [])
    )
    for g in ensure_list(group_list):
        name = get_road_name_from_location(g)
        if name:
            return name

    # Case C: point locations on linear reference
    point = loc_ref.get("locPointLocation")
    if point:
        for pl in ensure_list(point.get("pointAlongLinearElement")):
            try:
                name = (
                    pl.get("linearElement", {})
                      .get("locLinearElementByCode", {})
                      .get("roadName")
                )
                if name:
                    return name
            except:
                pass

    return None


# --------------------------------------------------------------
# 2. Extract number of lanes closed
# --------------------------------------------------------------
def get_lanes_restricted(loc_ref):

    # Case A: locLinearLocation -> supplementaryPositionalDescription
    lin = loc_ref.get("locLinearLocation")
    if lin:
        sup = lin.get("supplementaryPositionalDescription", {})
        car = ensure_list(sup.get("carriageway", []))

        for c in car:
            impact = (
                c.get("carriagewayExtensionG", {})
                  .get("impactOnCarriageway", {})
                  .get("numberOfLanesRestricted")
            )
            if impact is not None:
                return impact

    # Case B: grouped locations
    group_list = (
        loc_ref.get("locLocationGroupByList", {})
              .get("locationContainedInGroup", [])
    )
    for g in ensure_list(group_list):
        impact = get_lanes_restricted(g)
        if impact:
            return impact

    # Case C: fallback using lane-level closure counts
    if lin:
        sup = lin.get("supplementaryPositionalDescription", {})
        car = ensure_list(sup.get("carriageway", []))

        for c in car:
            lanes = ensure_list(c.get("lane", []))
            closed_count = 0

            for ln in lanes:
                status = (
                    ln.get("laneExtensionG", {})
                      .get("impactOnLanes", {})
                      .get("impactExtensionG", {})
                      .get("lanesStatus")
                )
                if str(status).lower() == "closed":
                    closed_count += 1

            if closed_count > 0:
                return closed_count

    return None

In [8]:
# --------------------------------------------------------------
# Extract coordinates (lat, lon) from posList
# --------------------------------------------------------------
def get_closure_coordinates(loc_ref):
    coords = []

    # Case A: locLinearLocation
    lin = loc_ref.get("locLinearLocation")
    if lin:
        gml = (
            lin.get("gmlLineString", {})
               .get("locGmlLineString", {})
               .get("posList")
        )
        if gml:
            nums = list(map(float, gml.split()))
            pairs = [(nums[i], nums[i+1]) for i in range(0, len(nums), 2)]
            coords.extend(pairs)

    # Case B: multiple locations
    group_list = (
        loc_ref.get("locLocationGroupByList", {})
              .get("locationContainedInGroup", [])
    )
    for g in ensure_list(group_list):
        coords.extend(get_closure_coordinates(g))

    return coords

In [9]:
def get_poslist(loc_ref):
    poslists = []

    # A. Single linear location
    lin = loc_ref.get("locLinearLocation")
    if lin:
        gml = (lin.get("gmlLineString", {})
                  .get("locGmlLineString", {})
                  .get("posList"))
        if gml:
            poslists.append(gml)

    # B. Multiple segments
    group_list = (
        loc_ref.get("locLocationGroupByList", {})
              .get("locationContainedInGroup", [])
    )
    for g in ensure_list(group_list):
        gml = get_poslist(g)
        if gml:
            if isinstance(gml, list):
                poslists.extend(gml)
            else:
                poslists.append(gml)

    return poslists if poslists else None

In [10]:
from datetime import datetime
import pandas as pd

# =========================
# MAIN EXTRACTION + SAVE
# =========================
def extract_road_closures(data_dict):
    all_records = []
    chunk_counter = 0
    current_blob = generate_blob_name(closureType, params['startDateTime'])
    situations = ensure_list(data_dict.get('D2Payload', {}).get('situation', []))
    print(situations,"situations")
    for s in situations:
        situation_id = s.get('idG')
        situation_records = ensure_list(s.get('situationRecord'))

        for rec in situation_records:

            if not isinstance(rec, dict):
                continue

            sm = rec.get('sitRoadOrCarriagewayOrLaneManagement', {})

            validity = sm.get('validity', {})
            time_spec = validity.get('validityTimeSpecification', {})
            cause = sm.get('cause', {})
            loc_ref = sm.get('locationReference', {})

            # Extract fields
            road_name = get_road_name_from_location(loc_ref)
            lanes_closed = get_lanes_restricted(loc_ref)
            poslist = get_poslist(loc_ref)
            coords = get_closure_coordinates(loc_ref)

            # Compute centroid
            if coords:
                lats = [c[0] for c in coords]
                lons = [c[1] for c in coords]
                closure_lat = sum(lats) / len(lats)
                closure_lon = sum(lons) / len(lons)
            else:
                closure_lat = None
                closure_lon = None

            # Create record
            record = {
                "situation_id": situation_id,
                "record_id": sm.get("idG"),
                "start_time": time_spec.get("overallStartTime"),
                "end_time": time_spec.get("overallEndTime"),
                "validity_status": validity.get("validityStatus"),
                "cause_type": cause.get("causeType"),
                "source": sm.get("source", {}).get("sourceIdentification"),
                "road_name": road_name,
                "lanes_closed": lanes_closed,
                "closure_lat": closure_lat,
                "closure_lon": closure_lon,
                "poslist": poslist,

                # Added (important for consistency)
                "ingestion_time": datetime.utcnow()
            }

            # buffer.append(record)
            all_records.append(record)

            # =========================
            # BATCH SAVE (Darwin style)
            # =========================

            if len(all_records) >= BATCH_SIZE:
                upload_to_azure(all_records, current_blob)

                chunk_counter += len(all_records)
                all_records.clear()

                # File rotation
                if chunk_counter >= UPLOAD_SIZE:
                    print("Chunk limit reached → rotating file")
                    current_blob = generate_blob_name(closureType, params['startDateTime'])
                    chunk_counter = 0

    # Save remaining records
    if all_records:
        upload_to_azure(all_records, current_blob)
        print(f"Saved {len(all_records)} records")

    print("Road closures extraction + save completed")
    return pd.DataFrame(all_records)

In [11]:
road_closures_df = extract_road_closures(road_closures_dict)


[{'idG': '438180', 'situationVersionTime': '2026-05-28T10:03:41.41Z', 'headerInformation': {'confidentiality': 'noRestriction', 'informationStatus': 'real'}, 'situationRecord': {'sitRoadOrCarriagewayOrLaneManagement': {'idG': '1-1768307239-c75618af-c31e-4947-94c0-fc5400282af5', 'versionG': '4', 'situationRecordCreationTime': '2026-01-13T08:37:38.38Z', 'situationRecordVersionTime': '2026-05-28T10:03:41.41Z', 'probabilityOfOccurrence': 'probable', 'complianceOption': 'mandatory', 'roadOrCarriagewayOrLaneManagementType': {'value': 'carriagewayClosures'}, 'source': {'sourceIdentification': 'roadworks'}, 'validity': {'validityStatus': 'planned', 'validityTimeSpecification': {'overallStartTime': '2026-01-12T09:00:00Z', 'overallEndTime': '2027-09-03T22:00:00Z'}}, 'cause': {'causeType': 'roadMaintenance', 'detailedCauseType': {'roadMaintenanceType': 'roadworks'}}, 'generalPublicComment': {'comment': 'M65 East and Westbound Jct 5 slips and approach roads - 24 hr narrow lanes, speed restrictions

In [12]:
print("Shape:", road_closures_df.shape)
road_closures_df.head()

Shape: (97, 13)


,situation_id,record_id,start_time,end_time,validity_status,cause_type,source,road_name,lanes_closed,closure_lat,closure_lon,poslist,ingestion_time
0,438180,1-1768307239-c75618af-c31e-4947-94c0-fc5400282af5,2026-01-12T09:00:00Z,2027-09-03T22:00:00Z,planned,roadMaintenance,roadworks,M65,3,53.727002,-2.444131,[53.727379 -2.443361 53.727713 -2.442955 53.72...,2026-05-28 10:22:53.939340
1,403789,1-1742947970-81c20dc7-ac9a-4cf6-b49f-fba44ceb29ca,2025-01-19T19:54:00Z,2027-12-31T06:00:00Z,active,constructionWork,roadworks,A57,0,53.458529,-2.009325,[53.458871 -2.008071 53.458911 -2.007984 53.45...,2026-05-28 10:22:53.939340
2,403789,1-1743814486-584effb6-f545-483a-b7d1-f3daff9bb6ee,2025-01-19T19:54:00Z,2027-12-31T06:00:00Z,active,constructionWork,roadworks,A57,1,53.458529,-2.009325,"[53.457546 -2.011857 53.457731 -2.011458, 53.4...",2026-05-28 10:22:53.939340
3,422047,1-1744065911-fd8bbeea-aada-40f9-93f4-c0768334412b,2025-04-13T11:00:00Z,2026-06-30T05:00:00Z,planned,roadMaintenance,roadworks,M6,1,54.649928,-2.752951,[54.650514 -2.754769 54.650106 -2.753878 54.64...,2026-05-28 10:22:53.939340
4,422047,1-1744065911-59ead3d8-3a69-426d-9d9e-b008be4a6c04,2025-04-13T05:00:00Z,2026-05-30T05:00:00Z,planned,roadMaintenance,roadworks,M6,1,54.651055,-2.755419,[54.653028 -2.758548 54.652922 -2.758552 54.65...,2026-05-28 10:22:53.939340
